In [1]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
!wget $data

--2026-10-13 08:47:34--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.003s  

2026-10-13 08:47:34 (78.5 MB/s) - ‘course_lead_scoring_2026.csv’ saved [278746/278746]



In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [ ]:
## Data Preparation ##


In [ ]:
## Accuracy and Dummy Model ##
# move threshold to increase accuracy
thresholds = np.linspace(0, 1, 21)
scores = []
for t in thresholds:
    churn_decision = (y_pred >= t)
    score = (y_val == churn_decision).mean()
    print('%.2f %.3f' % (t, score))
    scores.append(score)

from sklearn.metrics import accuracy_score
accuracy(y_val, y_pred >= 0.5)

plt.plot(thresholds, scores)

from collections import Counter

Counter(y_pred >= 1.0)

In [ ]:
## Confusion Table ##
# Accuracy without being affected by Class Imbalance
tp = (predict_positive & actual_positive).sum()
tn = (predict_negative & actual_negative).sum()
fp = (predict_positive & actual_negative).sum()
fn = (predict_negative & actual_positive).sum()

confusion_matrix = np.array([
    [tn, fp],
    [fn, tp]
])

(confusion_matrix / confusion_matrix.sum()).round(2)

In [ ]:
## Precision and Recall ##
(tp + tn) / (tp + tn + fp + fn)
precision = tp / tp + fp
recall = tp / tp + fn

In [ ]:
## ROC Curves ##
tpr = tp / (tp + fn)
fpr = fp / (fp + tn)

In [ ]:
thresholds = np.linspace(0, 1, 101)

for t in thresholds:
    actual_positive = (y_val == 1)
    actual_negative = (y_val == 0)

    predict_positive = (y_pred >= t)
    predict_negative = (y_pred < t)

    tp = (predict_positive & actual_positive).sum()
    tn = (predict_negative & actual_negative).sum()

    fp = (predict_positive & actual_negative).sum()
    fn = (predict_negative & actual_positive).sum()

    scores.append((t, tp, fp, fn, tn))

In [ ]:
df_scores = pd.DataFrame(scores, columns=)

df_scores['tpr'] = df_scores.tp / (df_scores.tp + df_scores.fn)
df_scores['fpr'] = df_scores.fp / (df_scores.fp + df_scores.tn)

plt.plot(df_scores.threshold, df_scores['tpr'], label='TPR')
plt.plot(df_scores.threshold, df_scores['fpr'], label='FPR')
plt.legend()

In [ ]:
## Random model ##
np.random.seed(1)
y_rand = np.random.uniform(0, 1, size=len(y_val))
((y_rand >= 0.5) == y_val).mean()

In [ ]:
def tpr_fpr_dataframe(y_val, y_pred):
    scores = []
    thresholds = np.linspace(0, 1, 101)

    for t in thresholds:
        actual_positive = (y_val == 1)
        actual_negative = (y_val == 0)
    
        predict_positive = (y_pred >= t)
        predict_negative = (y_pred < t)
    
        tp = (predict_positive & actual_positive).sum()
        tn = (predict_negative & actual_negative).sum()
    
        fp = (predict_positive & actual_negative).sum()
        fn = (predict_negative & actual_positive).sum()
    
        scores.append((tp, fp, fn, tn))

    columns = ['threshold', 'tp', 'fp', 'fn', 'tn']
    df_scores = pd.DataFrame(scores, columns=columns)

    df_scores['tpr'] = df_scores.tp / (df_scores.tp + df_scores.fn)
    df_scores['fpr'] = df_scores.fp / (df_scores.fp + df_scores.tn)
    
    plt.plot(df_scores.threshold, df_scores['tpr'], label='TPR')
    plt.plot(df_scores.threshold, df_scores['fpr'], label='FPR')
    plt.legend()

    return df_scores
    

In [ ]:
df_rand = tpr_fpr_dataframe(y_val, y_rand)

In [ ]:
## Ideal Model ##
num_neg = (y_val == 0).sum()
num_pos = (y_val == 1).sum()
num_neg, num_pos

In [ ]:
y_ideal = np.repeat([0, 1], [num_neg, num_pos])
y_ideal

In [ ]:
y_ideal_pred = np.linspace(0, 1, len(y_val))

In [ ]:
((y_ideal_pred >= 0.729) == y_ideal).mean()

In [ ]:
df_ideal = tpr_fpr_dataframe(y_ideal, y_ideal_pred)
df_ideal[::10]

In [ ]:
plt.plot(df_ideal.threshold, df_ideal['tpr'], label='TPR')
plt.plot(df_ideal.threshold, df_ideal['fpr'], label='FPR')
plt.legend()

In [ ]:
from sklearn.metrics import roc_curve, auc
fpr, tpr, thresholds = roc_curve(y_val, y_pred)

In [ ]:
## Area under TPR-FPR: ROC ##
# (good for binary classification model evaluations)
auc(fpr, tpr)
auc(df_scores.fpr, df_scores.tpr)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_val, y_pred)
auc(df_scores.fpr, df_scores.tpr)

In [ ]:
from sklearn.metrics import roc_auc_score # area under curve
roc_auc_score(y_val, y_pred)
neg = y_pred[y_val == 0]
pos = y_pred[y_val == 1]

import random
n = 10000
success = 0
for i in range(n):
    pos_ind = random.randint(0, len(pos) - 1)
    neg_ind = random.randint(0, len(neg) - 1)
    
    if pos[pos_ind] > neg[neg_ind]:
        success += 1
success / n

In [ ]:
n = 50000

np.random.seed(1)
pos_ind = np.random.randint(0,len(pos), size=n)
neg_ind = np.random.randint(0,len(pos), size=n)

(pos[pos_ind] > neg[neg_ind]).mean()

In [ ]:
pos[pos_ind] > neg[neg_ind]

In [ ]:
## Cross Validation ##
# fine-tuning to select best parameters
# k-fold cross validation

def train(df, y):
    dicts = df[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    model = LogisticRegression()
    model.fit(X_train, y_train)

    return dv, model

dv, model = train(df_train, y_train)

def predict(df, dv, model):
    dicts = df[categorical + numerical].to_dict(orient='records')

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

y_pred = predict(df_val, dv, model)
    

In [ ]:
from sklearn.model_selection import KFold
kfold = KFold(n_splits=3, shuffle=True, random_state=1)
train_idx, val_idx = next(kfold.split(df_full_train))

len(train_idx), len(val_idx)
len(df_full_train)
df_train = df_full_train.iloc[train_idx]
df_val = df_full_train.iloc[val_idx]

In [ ]:
!pip install tqdm

In [2]:
import tqdm

scores = []
n_splits = 5

for C in tqdm([1.0]):
    kfold = KFold(n_splits=n_splits, shuffle=True, random_states=1)
    scores = []
    for train_idx, val_idx in tqdm(kfold.split(df_full_train)):
        df_train = df_full_train.iloc[train_idx]
        df_val = df_full_train.iloc[val_idx]
    
        y_train = df_train.churn.values
        y_val = df_val.churn.values
    
        dv, model = train(df_train, y_train, C=C)
        y_pred = predict(df_val, dv, model)
    
        auc = roc_auc_score(y_val, y_pred)
        scores.append(auc)
    print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))

TypeError: 'module' object is not callable. Did you mean: 'tqdm.tqdm(...)'?

In [ ]:
dv, model = train(df_full_train, df_full_train.churn.values, C=1.0)
y_pred = predict(df_val, dv, model)

auc = roc_auc_score(y_test, y_pred)
auc